In [1]:
import os
import json
import numpy as np
from typing import List, Tuple
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
def bow_from_scratch(corpus: List[str]) -> Tuple[List[str], np.ndarray]:
    """Generates vocabulary and Bag-of-Words matrix from scratch."""
    vocab = sorted(list(set(w.lower() for doc in corpus for w in doc.split())))
    word_to_idx = {w: i for i, w in enumerate(vocab)}
    
    matrix = np.zeros((len(corpus), len(vocab)), dtype=int)
    for row_idx, doc in enumerate(corpus):
        for word in doc.split():
            w_clean = word.lower()
            if w_clean in word_to_idx:
                matrix[row_idx, word_to_idx[w_clean]] += 1
    return vocab, matrix

def tfidf_from_scratch(corpus: List[str]) -> Tuple[List[str], np.ndarray]:
    """Generates normalized TF-IDF feature matrix from scratch."""
    vocab, bow = bow_from_scratch(corpus)
    N = len(corpus)
    
    # Term Frequency (TF)
    doc_lengths = bow.sum(axis=1, keepdims=True)
    tf = bow / np.maximum(doc_lengths, 1)
    
    # Inverse Document Frequency (IDF)
    df = (bow > 0).sum(axis=0)
    idf = np.log((N + 1) / (df + 1)) + 1
    
    tfidf = tf * idf
    
    # L2 Normalization
    norms = np.linalg.norm(tfidf, axis=1, keepdims=True)
    tfidf_normalized = np.where(norms > 0, tfidf / norms, tfidf)
    return vocab, tfidf_normalized

def run_plagiarism_check(corpus: List[str], doc_names: List[str], threshold: float = 0.70):
    """Calculates cosine similarity across document pairs to flag potential plagiarism."""
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(corpus).toarray()
    sim_matrix = cosine_similarity(tfidf_matrix)

    print(f"{'Document A':<20} | {'Document B':<20} | {'Cosine Sim':<12} | {'Status':<15}")
    print("-" * 75)

    for i in range(len(doc_names)):
        for j in range(i + 1, len(doc_names)):
            score = sim_matrix[i, j]
            status = "FLAGGED PLAGIARISM" if score >= threshold else "CLEAN"
            print(f"{doc_names[i]:<20} | {doc_names[j]:<20} | {score:.4f}       | {status:<15}")

def main():
    print("=" * 60)
    print("ASSIGNMENT 3: BAG-OF-WORDS & TF-IDF VECTORIZATION")
    print("=" * 60)

    data_path = r"D:\NLP\academic_submissions.json"
    if not os.path.exists(data_path):
        print(f"Data file not found at {data_path}")
        return

    with open(data_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    corpus = [item["content"] for item in data]
    doc_ids = [item["id"] for item in data]

    # 1. From-Scratch Verification
    vocab_scratch, tfidf_scratch = tfidf_from_scratch(corpus)
    print(f"\nVocabulary Size (From Scratch): {len(vocab_scratch)} words")
    print(f"TF-IDF Matrix Shape: {tfidf_scratch.shape}")

    # 2. Scikit-learn Comparison
    sklearn_vec = TfidfVectorizer()
    tfidf_sklearn = sklearn_vec.fit_transform(corpus).toarray()
    print(f"Scikit-Learn TF-IDF Matrix Shape: {tfidf_sklearn.shape}")

    # 3. Academic Plagiarism Detection Module
    print("\n--- Academic Integrity Plagiarism Audit Report ---")
    run_plagiarism_check(corpus, doc_ids, threshold=0.70)

if __name__ == "__main__":
    main()


ASSIGNMENT 3: BAG-OF-WORDS & TF-IDF VECTORIZATION

Vocabulary Size (From Scratch): 7 words
TF-IDF Matrix Shape: (3, 7)
Scikit-Learn TF-IDF Matrix Shape: (3, 5)

--- Academic Integrity Plagiarism Audit Report ---
Document A           | Document B           | Cosine Sim   | Status         
---------------------------------------------------------------------------
paper_1.txt          | submission_A.txt     | 0.6158       | CLEAN          
paper_1.txt          | paper_2.txt          | 0.4774       | CLEAN          
submission_A.txt     | paper_2.txt          | 0.4774       | CLEAN          


In [3]:
import os, json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
path = r"D:\NLP\academic_submissions.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

docs = [x["content"] for x in data]
ids = [x["id"] for x in data]

print("Documents:", len(docs))

Documents: 3


In [6]:
def make_bow(docs):
    words = sorted({w.lower() for text in docs for w in text.split()})
    index = {w: i for i, w in enumerate(words)}
    matrix = np.zeros((len(docs), len(words)), dtype=int)

    for r, text in enumerate(docs):
        for word in text.lower().split():
            matrix[r, index[word]] += 1

    return words, matrix

vocab, bow = make_bow(docs)

print("Vocabulary:", len(vocab))
print("BOW Shape:", bow.shape)

Vocabulary: 7
BOW Shape: (3, 7)


In [7]:
def make_tfidf(bow):
    tf = bow / np.maximum(bow.sum(axis=1, keepdims=True), 1)
    df = np.count_nonzero(bow, axis=0)
    idf = np.log((len(bow) + 1) / (df + 1)) + 1

    result = tf * idf
    norm = np.linalg.norm(result, axis=1, keepdims=True)

    return result / np.maximum(norm, 1e-12)

tfidf_manual = make_tfidf(bow)

print("TF-IDF Shape:", tfidf_manual.shape)

TF-IDF Shape: (3, 7)


In [8]:
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(docs)

similarity = cosine_similarity(vectors)

for i in range(len(ids)):
    for j in range(i + 1, len(ids)):
        score = similarity[i, j]
        status = "FLAGGED" if score >= 0.70 else "CLEAN"

        print(f"{ids[i]} | {ids[j]} | {score:.4f} | {status}")

paper_1.txt | submission_A.txt | 0.6158 | CLEAN
paper_1.txt | paper_2.txt | 0.4774 | CLEAN
submission_A.txt | paper_2.txt | 0.4774 | CLEAN


In [9]:
print("\n" + "=" * 75)
print("        ACADEMIC INTEGRITY PLAGIARISM AUDIT REPORT")
print("=" * 75)

print(f"Total Documents       : {len(docs)}")
print(f"Vocabulary Size        : {len(vocab)}")
print(f"TF-IDF Matrix Shape    : {tfidf_manual.shape}")
print(f"Similarity Threshold   : 0.70")

print("\n" + "-" * 75)
print(f"{'Document A':<18} {'Document B':<18} {'Similarity':<12} {'Status'}")
print("-" * 75)

for i in range(len(ids)):
    for j in range(i + 1, len(ids)):
        score = similarity[i, j]
        status = "FLAGGED" if score >= 0.70 else "CLEAN"

        print(
            f"{ids[i]:<18} "
            f"{ids[j]:<18} "
            f"{score:<12.4f} "
            f"{status}"
        )

print("-" * 75)
print("Report Generation Completed")
print("=" * 75)


        ACADEMIC INTEGRITY PLAGIARISM AUDIT REPORT
Total Documents       : 3
Vocabulary Size        : 7
TF-IDF Matrix Shape    : (3, 7)
Similarity Threshold   : 0.70

---------------------------------------------------------------------------
Document A         Document B         Similarity   Status
---------------------------------------------------------------------------
paper_1.txt        submission_A.txt   0.6158       CLEAN
paper_1.txt        paper_2.txt        0.4774       CLEAN
submission_A.txt   paper_2.txt        0.4774       CLEAN
---------------------------------------------------------------------------
Report Generation Completed
